# grpo-lab: ARC-AGI pilot on a rented GPU

Minimal driver for `scripts/run_arc_experiment.sh` on Colab (A100) or a Lambda / RunPod Jupyter box.
Everything is the repo's own scripts; this notebook only clones, installs, picks a config and tails the log.

Pick the config for the card you got:

| GPU | config |
|---|---|
| >= 16 GB (A100, L4 24 GB, RTX 3080 16 GB) | `configs/qwen2.5-1.5b-arc-pilot.yaml` |
| 8 GB, or a 15 GB T4 | `configs/qwen2.5-1.5b-arc-pilot-8gb.yaml` |

The full pipeline (baseline evals, 100 GRPO steps, post evals, official ARC-1 eval before/after) is a
multi-hour job: ~90 s/step on an RTX 4060, much less on an A100. On Colab use Pro+ background execution
or expect the session to be reclaimed; results are synced to Drive at the end if Drive is mounted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, subprocess
GPU_MB = int(subprocess.check_output(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"]).split()[0])
CONFIG = "configs/qwen2.5-1.5b-arc-pilot.yaml" if GPU_MB >= 16000 else "configs/qwen2.5-1.5b-arc-pilot-8gb.yaml"
print("using", CONFIG)

In [ ]:
# Clone + install the pinned environment (uv.lock: torch 2.11 / CUDA 12.8). ~3-5 min.
import os
if not os.path.exists("grpo-lab"):
    !git clone https://github.com/gian-g3dai/grpo-lab
%cd grpo-lab
!pip install -q uv
!bash scripts/setup.sh
!bash scripts/get_arc_data.sh
!.venv/bin/python -m pytest tests -q

In [ ]:
# Optional: persist results to Google Drive when running on Colab.
DRIVE = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/grpo-lab"
    os.makedirs(DRIVE, exist_ok=True)
except ImportError:
    print("not on Colab; results stay in ./results and ./outputs")

In [ ]:
# Run the whole pipeline. On Colab the output streams here; on a Jupyter box you can also
# run it with nohup from a terminal and tail outputs/<run_name>/pipeline.log instead.
import yaml
RUN = yaml.safe_load(open(CONFIG))["run_name"]
os.makedirs(f"outputs/{RUN}", exist_ok=True)
!bash scripts/run_arc_experiment.sh {CONFIG} 2>&1 | tee outputs/{RUN}/pipeline.log | grep --line-buffered -E "^==|^\[|'reward'|Traceback|Error|saved to"


In [ ]:
# Summary of every eval file the pipeline wrote (pass@1 / pass@2 / official task score).
import glob, json
for f in sorted(glob.glob(f"results/{RUN}/*.json")):
    s = json.load(open(f))
    print(f"{os.path.basename(f):22s} n={s['n']:4d} pass1={s['pass1']:.3f} pass2={s['pass2']:.3f} task_score={s['task_score']:.3f} cell_acc={s['cell_acc']:.3f}")
if DRIVE:
    !cp -r results/{RUN} {DRIVE}/ && cp -r outputs/{RUN}/final {DRIVE}/{RUN}-adapter && echo synced to Drive